# 04 Train Probability Model

Train an Elo-only probability baseline and a logistic regression model on engineered features.

In [ ]:
import json
from pathlib import Path

import joblib
import pandas as pd
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


def resolve_artifacts_dir() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd / "artifacts", cwd.parent / "artifacts"]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return cwd.parent / "artifacts" if cwd.name == "notebooks" else cwd / "artifacts"


ARTIFACTS_DIR = resolve_artifacts_dir()
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

X_train = pd.read_csv(ARTIFACTS_DIR / "train_features.csv")
X_valid = pd.read_csv(ARTIFACTS_DIR / "valid_features.csv")
y_train = pd.read_csv(ARTIFACTS_DIR / "train_labels.csv")["blue_team_win"]
y_valid = pd.read_csv(ARTIFACTS_DIR / "valid_labels.csv")["blue_team_win"]


In [ ]:
def build_training_frame(df: pd.DataFrame) -> pd.DataFrame:
    return df.drop(columns=["date", "blue_team", "red_team"], errors="ignore")


def train_logistic_regression_local(X_train_df: pd.DataFrame, y_train_series: pd.Series) -> Pipeline:
    numeric_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )
    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, make_column_selector(dtype_include=["number"])),
        ]
    )
    model = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("classifier", LogisticRegression(max_iter=1000)),
        ]
    )
    model.fit(X_train_df, y_train_series.astype(int))
    return model


In [ ]:
train_frame = build_training_frame(X_train)
valid_frame = build_training_frame(X_valid)
model = train_logistic_regression_local(train_frame, y_train)
valid_prob = model.predict_proba(valid_frame)[:, 1]


In [ ]:
joblib.dump(model, ARTIFACTS_DIR / "logistic_regression_model.joblib")
pd.DataFrame(
    {
        "blue_win_prob": valid_prob,
        "blue_team_win": y_valid.astype(int),
    }
).to_csv(ARTIFACTS_DIR / "validation_predictions.csv", index=False)
(ARTIFACTS_DIR / "train_manifest.json").write_text(
    json.dumps(
        {
            "model_name": "logistic_regression",
            "feature_columns": list(train_frame.columns),
            "train_rows": int(len(train_frame)),
            "valid_rows": int(len(valid_frame)),
        },
        indent=2,
    )
)

pd.DataFrame(
    {
        "blue_win_prob": valid_prob,
        "blue_team_win": y_valid.astype(int).reset_index(drop=True),
    }
).head()
